# Profitability visuals

Simulates the **live pipeline** on the **test** coins and makes the charts for the README.

**Same as the live bot**
* **Strategy:** the strategy file `strategy_<name>.json` that `strategy.ipynb`'s export cell writes, copied by hand into `live_pipeline/artifacts/`: the parameters, the combo cutoff and the simulation settings, the same files the live pipeline reads. `STRATEGY_NAME` picks the file (`None`: the first strategy file there, as the live pipeline's default).
* **Models and score:** the move and direction models from `live_pipeline/artifacts/`, scored as `live_pipeline/scoring.py` does. They must be the models the strategy was tuned on: the notebook checks their fingerprints against the strategy file. The buy score is the raw combo, P(move) × P(direction), and the gate is `combo ≥ combo_cutoff`.
* **Decisions:** the decision functions of `live_pipeline/strategy.py`, copied into this notebook unchanged and called minute by minute as `bot_loop.py` calls them (entry filters, re-entry threshold, the state machine, rebases in a race). If they change in `strategy.py`, copy them here again.

The notebook reads no Python files of the project: everything it runs is in this notebook; it reads only data, the models and the strategy file.
* **Fills:** paper fills, slippage per side from `FIXED_PARAMS`; each buy is capped by the liquidity estimate and `desired_price_impact_pct`.

**As in the research, so the numbers compare with `strategy.ipynb`** (the settings come from the strategy file)
* $20 per coin, at $130 per SOL.
* A position still open at the end of a coin's data is sold at its last close.
* The **capped score** is each coin's final capital capped at 10 × the start ($200), the score `strategy.ipynb` optimises and reports.

* No buys in a coin's first 12 hours (`trading_start_minutes`), as `strategy.ipynb`'s simulator, so the capped score reproduces the strategy's test score. The live bot has no such start (set `TRADING_START_MINUTES = 0` after the config to simulate that).

**Folders:** the models and the strategy file from `live_pipeline/artifacts/`, the test split and the cleaned OHLCV from `data/` (`dataset_v9/splits/test.csv`, `processed_data/`); the charts and the score cache go to `research_development/readme_visuals/`.

In [ ]:
# ============================================================================
# CONFIG
# ============================================================================
import warnings
from pathlib import Path

import json

import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.simplefilter('ignore', pd.errors.PerformanceWarning)

# ── INPUTS ───────────────────────────────────────────────────────────────────
# ── REPOSITORY FOLDERS ───────────────────────────────────────────────────────
# All paths are relative to the repository's top folder, found from where the notebook runs:
#   data/                         raw_data/, processed_data/, dataset_v9/        (not committed)
#   research_development/<name>/  each notebook's outputs                        (not committed)
#   live_pipeline/artifacts/      the chosen models and strategy files, copied there by hand
from pathlib import Path


def _repo_root():
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / 'live_pipeline').is_dir() and (p / 'development_notebooks').is_dir():
            return p
    raise RuntimeError("run this notebook inside the repository: no folder above it holds "
                       "live_pipeline/ and development_notebooks/")


REPO = _repo_root()
REPO_DATA = REPO / 'data'
REPO_RESULTS = REPO / 'research_development'
REPO_ARTIFACTS = REPO / 'live_pipeline' / 'artifacts'

MODEL_DIR = REPO_ARTIFACTS                  # one move + one direction .ubj (+ their .json) and the strategy files
TEST_SPLIT = REPO_DATA / 'dataset_v9' / 'splits' / 'test.csv'
OHLCV_FOLDER = REPO_DATA / 'processed_data'  # cleaned OHLCV (data_cleaning.ipynb)

# ── OUTPUT ───────────────────────────────────────────────────────────────────
OUT_DIR = REPO_RESULTS / 'readme_visuals'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── STRATEGY (written by strategy.ipynb's export cell) ───────────────────────
STRATEGY_NAME = None                        # strategy_<name>.json in MODEL_DIR; None = the first one there (by file name)
CAP_X_DEFAULT = 10                          # used if the file has no score_capital_ceiling_x

_files = {p.stem[len('strategy_'):]: p for p in sorted(MODEL_DIR.glob('strategy_*.json'))}
if STRATEGY_NAME is None:
    if not _files:
        raise RuntimeError(f"no strategy_<name>.json in {MODEL_DIR}/: copy one from research_development/strategy_results/")
    STRATEGY_NAME = next(iter(_files))          # the first by file name, as the live pipeline's default
elif STRATEGY_NAME not in _files:
    raise RuntimeError(f"no strategy_{STRATEGY_NAME}.json in {MODEL_DIR}/ (found: {sorted(_files) or 'none'})")
with open(_files[STRATEGY_NAME]) as _f:
    STRATEGY = json.load(_f)
if STRATEGY.get('format') != 'bot-strategy/1':
    raise RuntimeError(f"unknown strategy file format {STRATEGY.get('format')!r}")

P = dict(STRATEGY['params'], combo_cutoff=float(STRATEGY['combo_cutoff']))   # the strategy's parameters
SIM = STRATEGY['simulation']
F = {k: SIM[k] for k in ('slippage_pct', 'desired_price_impact_pct', 'target_fill_pct', 'exit_mode')}
INITIAL_CAPITAL_USD = float(SIM['initial_capital_usd'])
SOL_PRICE_USD = float(SIM['sol_price_usd'])
CAP_X = SIM.get('score_capital_ceiling_x', CAP_X_DEFAULT)     # capped score: final capital capped at CAP_X x the start
TRADING_START_MINUTES = int(SIM['trading_start_minutes'])     # 720: no buys in a coin's first 12 hours
PRICE_POSITION_60M_FEATURE = 'price_position_1h'

_sel = STRATEGY.get('selection', {})
print(f"Strategy '{STRATEGY_NAME}': run {_sel.get('run')} of {_sel.get('n_runs')}, chosen by {_sel.get('selected_by')} "
      f"score (test ${_sel.get('test_score', float('nan')):.2f}, val ${_sel.get('val_score', float('nan')):.2f})")
for k, v in {**P, **F, 'trading_start_minutes': TRADING_START_MINUTES}.items():
    print(f'  {k:<32} {v}')

# ── CHARTS ───────────────────────────────────────────────────────────────────
TAIL = 0.20                     # chart 2: extra width after the simulation, as a share of the simulated period
CHART_SIZE = (12 * (1 + TAIL), 6.5)              # inches, both charts (saved at 200 dpi)
CHART_MARGINS = dict(left=0.065, right=0.985, top=0.90, bottom=0.13)   # same plot area in both charts

# ── CHART COLOURS (the dashboard's dark theme) ───────────────────────────────
THEME = {
    'figure': '#0f172a', 'plot': '#0a0e1a', 'text': '#e2e8f0', 'muted': '#94a3b8', 'grid': '#334155',
    'profit': '#10b981', 'loss': '#ef4444', 'start': '#cbd5e1', 'mean': '#60a5fa', 'median': '#facc15',
    'capital': '#667eea', 'realized': '#10b981', 'open': '#f59e0b', 'fees': '#ef4444',
}
# Matplotlib reads text between two '$' signs as math; tx() escapes them in titles and labels.
tx = lambda s: s.replace('$', r'\$')
plt.rcParams.update({
    'figure.facecolor': THEME['figure'], 'axes.facecolor': THEME['plot'], 'savefig.facecolor': THEME['figure'],
    'axes.edgecolor': THEME['grid'], 'axes.labelcolor': THEME['text'], 'text.color': THEME['text'],
    'xtick.color': THEME['muted'], 'ytick.color': THEME['muted'], 'grid.color': THEME['grid'],
    'grid.alpha': 0.6, 'axes.grid': True, 'axes.axisbelow': True, 'font.size': 11,
    'legend.facecolor': THEME['figure'], 'legend.edgecolor': THEME['grid'],
})

In [ ]:
# ============================================================================
# DATA: the test minutes scored by the two models, merged onto each coin's OHLCV
# ============================================================================
# As strategy.ipynb: every minute of the cleaned OHLCV is simulated; minutes
# without a dataset row (a coin's first 120 and last 30) get combo 0, no
# liquidity estimate and price positions of 1.0, so they never qualify.
# The models: exactly one move and one direction .ubj in MODEL_DIR, told apart by
# their _params.json ('model_type') or their file name (move* / dir*), as strategy.ipynb.
def _model_type(p):
    for pj in (p.with_name(p.stem + '_params.json'), p.with_suffix('.json')):   # move_params.json or move.json
        if pj.exists():
            t = json.load(open(pj)).get('model_type')
            if t in ('move', 'dir'):
                return t
    return 'move' if p.name.startswith('move') else ('dir' if p.name.startswith('dir') else None)


_found = {}
for p in sorted(MODEL_DIR.glob('*.ubj')):
    _found.setdefault(_model_type(p), []).append(p)
if len(_found.get('move', [])) != 1 or len(_found.get('dir', [])) != 1:
    raise RuntimeError(f'{MODEL_DIR}/ must hold exactly one move and one direction .ubj; found '
                       f'{ {k: [x.name for x in v] for k, v in _found.items()} }')
MODELS = {}
for k in ('move', 'dir'):
    m = xgb.Booster()
    m.load_model(str(_found[k][0]))
    MODELS[k] = m
    print(f'  {k:<5} {_found[k][0].name}  ({len(m.feature_names)} features)')


# the strategy's combo_cutoff is a score of the models it was tuned on: check their fingerprints
import hashlib


def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


for k, key in (('move', 'move'), ('dir', 'direction')):
    if _sha256(_found[k][0]) != STRATEGY['models'][key]['sha256']:
        raise RuntimeError(f"{_found[k][0].name} is not the {key} model strategy '{STRATEGY_NAME}' was tuned on "
                           f"({STRATEGY['models'][key].get('file')}); use the models it was exported with")
print(f"  models match strategy '{STRATEGY_NAME}'")


def predict(model, df):
    # as live_pipeline/scoring.py and strategy.ipynb: the model's own feature list, trees up to best_iteration
    f = list(model.feature_names)
    return model.predict(xgb.DMatrix(df[f], feature_names=f), iteration_range=(0, model.best_iteration + 1))


_sig = '|'.join(f'{p.name}:{p.stat().st_size}:{int(p.stat().st_mtime)}'
                for p in sorted(MODEL_DIR.glob('*.ubj'))) + f'|{TEST_SPLIT.stat().st_size}'
_cache = OUT_DIR / 'test_scores.pkl'
scores = None
if _cache.exists():
    cached = pd.read_pickle(_cache)
    if cached.attrs.get('sig') == _sig:
        scores = cached
if scores is None:
    print(f'Scoring the test split with the models in {MODEL_DIR}/ ...')
    fr = pd.read_csv(TEST_SPLIT)
    fr['coin_id'] = fr['coin_id'].astype(str)
    fr['timestamp'] = pd.to_datetime(fr['timestamp'], format='mixed', utc=True)
    pm, pdir = predict(MODELS['move'], fr), predict(MODELS['dir'], fr)
    scores = fr[['coin_id', 'timestamp', 'estimated_sol_1pct_impact', 'price_position_15m']].copy()
    scores['price_position_60m'] = fr[PRICE_POSITION_60M_FEATURE].to_numpy()
    scores['raw_move'] = pm.astype(np.float64)
    scores['raw_dir'] = pdir.astype(np.float64)
    scores['combo'] = (pm * pdir).astype(np.float64)   # the product in the models' float32, as the live bot
    scores.attrs['sig'] = _sig
    scores.to_pickle(_cache)
    del fr


def read_ohlcv(coin):
    # strategy.ipynb's _read_ohlcv: the cleaned file of a coin, prices in USD
    f = list(OHLCV_FOLDER.glob(f'{coin}_ohlcv_cleaned.csv')) or \
        list(OHLCV_FOLDER.glob(f"{coin.replace('_cleaned', '')}_ohlcv_cleaned.csv"))
    if not f:
        return None
    df = pd.read_csv(f[0])
    df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', utc=True, errors='coerce')
    df = df[df['timestamp'].notna()].sort_values('timestamp').reset_index(drop=True)
    for col in ['open', 'high', 'low', 'close']:
        df[col] = df[col] * SOL_PRICE_USD
    return df


merged, missing = {}, []
cols = ['timestamp', 'combo', 'raw_move', 'raw_dir', 'estimated_sol_1pct_impact', 'price_position_15m', 'price_position_60m']
for coin, grp in scores.groupby('coin_id', sort=True):
    o = read_ohlcv(coin)
    if o is None:
        missing.append(coin)
        continue
    m = o.merge(grp[cols], on='timestamp', how='left')
    m['combo'] = m['combo'].fillna(0.0)
    m['estimated_sol_1pct_impact'] = m['estimated_sol_1pct_impact'].fillna(0.0)
    m['price_position_15m'] = m['price_position_15m'].fillna(1.0)
    m['price_position_60m'] = m['price_position_60m'].fillna(1.0)
    merged[coin] = m
print(f'{len(merged)} test coins with OHLCV' + (f'; {len(missing)} without an OHLCV file: {missing[:5]}' if missing else ''))
print(f"minutes passing the combo cutoff ({P['combo_cutoff']:.4f}): "
      f"{(scores['combo'] >= P['combo_cutoff']).mean() * 100:.2f}% of scored test minutes")

## Strategy

The decision functions of `live_pipeline/strategy.py`, copied unchanged (the parameters are passed in).

In [ ]:
# ============================================================================
# STRATEGY — copied from live_pipeline/strategy.py
# ============================================================================
PHASE_NORMAL          = 'normal'
PHASE_STALE_RACE      = 'stale_race'
PHASE_LOSS_FLOOR_RACE = 'loss_floor_race'


def passes_entry_filters(combo, features, params):
    if combo is None or not (combo >= params['combo_cutoff']):
        return False
    pp15 = features.get('price_position_15m', 1.0)
    pp60 = features.get(PRICE_POSITION_60M_FEATURE, 1.0)
    pp15 = 1.0 if pp15 is None or pp15 != pp15 else pp15
    pp60 = 1.0 if pp60 is None or pp60 != pp60 else pp60
    if pp15 > params['max_price_position_15m']:
        return False
    if pp60 > params['max_price_position_60m']:
        return False
    return True


def effective_reentry_threshold(minutes_since_sell, params):
    re_thresh = params['reentry_threshold_pct']
    decay_start = float(params['reentry_decay_start_minutes'])
    if minutes_since_sell > decay_start:
        relaxation = (minutes_since_sell - decay_start) * params['reentry_decay_rate']
        effective = re_thresh - relaxation
        if effective < -5.0:
            return None
        return effective
    return re_thresh


def passes_reentry_threshold(current_close, current_idx, last_sell_close, last_sell_idx, params):
    if last_sell_close is None or last_sell_close <= 0 or last_sell_idx is None:
        return True
    effective = effective_reentry_threshold(float(current_idx - last_sell_idx), params)
    if effective is None:
        return True
    return current_close <= last_sell_close * (1.0 - effective)


def step_open_position(current_close, entry_price, minutes_held,
                       max_price_seen, below_floor_count, trade_phase, params):
    act_thresh = params['activation_threshold']
    stale_minutes = params['stale_trade_minutes']
    floor_pct = params['loss_floor_pct']
    n_consec_required = int(params['loss_floor_consecutive_minutes'])

    result = {
        'action': 'hold',
        'reason': '',
        'max_price_seen': max_price_seen,
        'below_floor_count': below_floor_count,
        'effective_activation_pct': act_thresh,
    }

    # ---- Update peak ----
    if max_price_seen is None or current_close > max_price_seen:
        max_price_seen = current_close
        result['max_price_seen'] = max_price_seen

    # ---- 1) ACTIVATION (highest priority, all phases) ----
    if max_price_seen >= entry_price * (1.0 + act_thresh):
        result['action'] = 'sell'
        result['reason'] = f'Activation reached (peak +{act_thresh * 100:.2f}%)'
        return result

    # ---- 2) Phase-specific transitions ----
    floor_price = entry_price * (1.0 - floor_pct)

    if trade_phase == PHASE_NORMAL:
        # 2a) Loss-floor arm-watch
        if current_close <= floor_price:
            below_floor_count += 1
            result['below_floor_count'] = below_floor_count
            if below_floor_count >= n_consec_required:
                result['action'] = 'enter_loss_floor_race'
                result['reason'] = (
                    f'Loss floor armed: {n_consec_required} consecutive minutes '
                    f'at/below -{floor_pct * 100:.2f}%'
                )
                return result
        else:
            below_floor_count = 0
            result['below_floor_count'] = 0

        # 2b) Stale (lowest priority)
        if minutes_held > stale_minutes:
            result['action'] = 'enter_stale_race'
            result['reason'] = f'Trade stale after {minutes_held} minutes'
            return result

    elif trade_phase == PHASE_LOSS_FLOOR_RACE:
        # Recovery-watch: the first minute above the floor disarms back to NORMAL
        if current_close > floor_price:
            result['action'] = 'recover_loss_floor'
            result['reason'] = (
                f'Loss floor recovered (close > entry * (1 - {floor_pct * 100:.2f}%))'
            )
            result['below_floor_count'] = 0
            return result
        # Stale still ticks during the loss-floor race -> terminal stale race
        if minutes_held > stale_minutes:
            result['action'] = 'enter_stale_race'
            result['reason'] = f'Trade stale after {minutes_held} minutes (during LF race)'
            return result

    # PHASE_STALE_RACE: only activation (above) or a re-entry ends it.
    return result


def compute_exit_price(current_close, slippage_pct):
    return current_close * (1.0 - slippage_pct)

## Simulation

One coin at a time, minute by minute, with the live bot's decision code (`bot_loop.py`'s order): a flat bot checks the entry filters and the re-entry threshold; an open position in a race first checks for a rebase, otherwise it takes one state-machine step. Each minute records the cash, the open position's value (shares × close) and the cumulative fees, the three series of the dashboard's capital chart.

In [ ]:
# ============================================================================
# SIMULATION — the live decision code, minute by minute
# ============================================================================
def simulate_live(m):
    close = m['close'].to_numpy(np.float64)
    combo = m['combo'].to_numpy(np.float64)
    pp15 = m['price_position_15m'].to_numpy(np.float64)
    pp60 = m['price_position_60m'].to_numpy(np.float64)
    liq = m['estimated_sol_1pct_impact'].to_numpy(np.float64)
    n = len(close)
    slip = F['slippage_pct']
    impact_x = F['desired_price_impact_pct'] * 100.0

    capital, shares, fees = INITIAL_CAPITAL_USD, 0.0, 0.0
    is_open = False
    last_sell_close, last_sell_idx = None, None
    trades = []
    cash_c, open_c, fee_c = np.empty(n), np.empty(n), np.empty(n)

    def sell(i, reason):
        nonlocal capital, shares, fees, is_open, last_sell_close, last_sell_idx
        exit_price = compute_exit_price(close[i], slip)
        capital += shares * exit_price
        fees += shares * close[i] * slip
        trades.append({'entry_idx': orig_idx, 'exit_idx': i, 'n_reentries': n_re, 'reason': reason,
                       'pnl_pct': (exit_price - orig_entry) / orig_entry})
        shares, is_open = 0.0, False
        last_sell_close, last_sell_idx = close[i], i

    for i in range(n):
        c = close[i]
        feats = {'price_position_15m': pp15[i], PRICE_POSITION_60M_FEATURE: pp60[i]}
        if not is_open:
            if (i >= TRADING_START_MINUTES and passes_entry_filters(combo[i], feats, P)
                    and passes_reentry_threshold(c, i, last_sell_close, last_sell_idx, P)):
                amount = min(capital, liq[i] * impact_x * SOL_PRICE_USD) if liq[i] > 0 else capital
                if amount > 0:
                    entry = c * (1.0 + slip)
                    shares = amount / entry
                    capital -= amount
                    fees += shares * c * slip
                    is_open = True
                    orig_entry, orig_idx, n_re = entry, i, 0
                    seg_entry, seg_idx = entry, i
                    max_seen, below, phase = c, 0, PHASE_NORMAL
        else:
            if (phase in (PHASE_STALE_RACE, PHASE_LOSS_FLOOR_RACE) and passes_entry_filters(combo[i], feats, P)
                    and passes_reentry_threshold(c, i, last_sell_close, last_sell_idx, P)):
                # rebase: new reference at this close, keep the shares
                seg_entry, seg_idx, max_seen, below, phase = c, i, c, 0, PHASE_NORMAL
                n_re += 1
            else:
                st = step_open_position(c, seg_entry, i - seg_idx, max_seen, below, phase, P)
                max_seen, below = st['max_price_seen'], st['below_floor_count']
                a = st['action']
                if a == 'sell':
                    sell(i, 'take_profit')
                elif a == 'enter_stale_race':
                    phase, below = PHASE_STALE_RACE, 0
                elif a == 'enter_loss_floor_race':
                    phase = PHASE_LOSS_FLOOR_RACE
                elif a == 'recover_loss_floor':
                    phase, below = PHASE_NORMAL, 0
        if is_open and i == n - 1:
            sell(i, 'end_of_data')       # as strategy.ipynb: close an open position at the last minute
        cash_c[i], open_c[i], fee_c[i] = capital, shares * c, fees

    return {'final_capital': capital, 'fees': fees, 'trades': trades,
            'timestamps': m['timestamp'].to_numpy(), 'cash': cash_c, 'open': open_c, 'cum_fees': fee_c}


results = {coin: simulate_live(m) for coin, m in merged.items()}

summary = pd.DataFrame([{
    'coin': coin, 'final_capital': r['final_capital'], 'trades': len(r['trades']),
    'rebases': sum(t['n_reentries'] for t in r['trades']), 'fees': r['fees'],
    'days': len(r['cash']) / 1440,
} for coin, r in results.items()])
summary['capped'] = summary['final_capital'].clip(upper=CAP_X * INITIAL_CAPITAL_USD)
# least to most profitable; 'rank' is a coin's position in both charts (1 = least profitable)
summary = summary.sort_values('final_capital', ascending=True, kind='stable').reset_index(drop=True)
summary.insert(0, 'rank', np.arange(1, len(summary) + 1))

capped_mean, capped_median = summary['capped'].mean(), summary['capped'].median()
n_up = int((summary['final_capital'] > INITIAL_CAPITAL_USD).sum())
print(f"trading start: {TRADING_START_MINUTES} min   coins: {len(summary)}   trades: {summary['trades'].sum()}")
print(f'capped score (mean):  ${capped_mean:.2f} per coin   median: ${capped_median:.2f}')
print(f"uncapped mean:        ${summary['final_capital'].mean():.2f} per coin")
print(f'coins above the $20 start: {n_up} of {len(summary)}')
print(f"fees paid: ${summary['fees'].sum():.2f} in total, ${summary['fees'].mean():.2f} per coin")
summary.head(10)

## Final capital per coin

Each test coin's capped final capital, least to most profitable, against the $20 start. The mean is the capped score; with many bots running in parallel on small amounts, a portfolio of coins earns the mean, even when the median coin does not.

In [ ]:
# ============================================================================
# CHART 1 — final capital per coin, least to most profitable
# ============================================================================
PLOT_KIND = 'bar'               # 'bar' or 'line'

vals = summary['capped'].to_numpy()        # summary is ordered least to most profitable
x = np.arange(1, len(vals) + 1)
start = INITIAL_CAPITAL_USD

fig, ax = plt.subplots(figsize=CHART_SIZE)
if PLOT_KIND == 'bar':
    colors = [THEME['profit'] if v > start else THEME['loss'] for v in vals]
    ax.bar(x, vals, width=0.85, color=colors, linewidth=0)
else:
    ax.plot(x, vals, color=THEME['text'], linewidth=1.6)
    ax.fill_between(x, start, vals, where=vals > start, color=THEME['profit'], alpha=0.35, interpolate=True)
    ax.fill_between(x, start, vals, where=vals <= start, color=THEME['loss'], alpha=0.35, interpolate=True)

ax.axhline(start, color=THEME['start'], linestyle='--', linewidth=1.3, label=tx(f'Start: ${start:.0f}'))
ax.axhline(capped_mean, color=THEME['mean'], linewidth=1.6, label=tx(f'Mean: ${capped_mean:.2f}'))
ax.axhline(capped_median, color=THEME['median'], linestyle=':', linewidth=1.8, label=tx(f'Median: ${capped_median:.2f}'))

ax.set_xlim(0.3, len(vals) + 0.7)
ax.set_ylim(0, max(vals.max(), capped_mean) * 1.06)
ax.set_xlabel('Test coins, least to most profitable')
ax.set_ylabel(tx(f'Final capital per coin (capped at ${CAP_X * start:.0f})'))
ax.set_title(tx(f'Final capital of ${start:.0f} per coin on {len(vals)} unseen test coins   '
                f'({n_up} of {len(vals)} end above the start)'), loc='left', fontsize=12.5, pad=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:.0f}'))
ax.grid(axis='x', visible=False)
ax.legend(loc='upper left', frameon=True)
fig.text(0.01, 0.005, tx(f'Strategy \'{STRATEGY_NAME}\', simulated with the live pipeline\'s decision code; {F["slippage_pct"] * 100:.1f}% slippage per side; '
         f'no buys in a coin\'s first {TRADING_START_MINUTES / 60:g} hours.'), fontsize=9, color=THEME['muted'])
fig.subplots_adjust(**CHART_MARGINS)
path = OUT_DIR / f'final_capital_per_coin_{PLOT_KIND}.png'
fig.savefig(path, dpi=200)
plt.show()
print(f'saved {path}')

## Capital over time for one coin

The dashboard's capital chart for one test coin: closed capital (up to the $20 start), realized profit above it, and the open position's value, with the cumulative fees in their own pane. `COIN_RANK` picks the coin in the same order as the first chart: `1` is the least profitable coin, `len(summary)` the most, and `COIN_RANK = 40` is the 40th bar of the first chart.

In [ ]:
# ============================================================================
# CHART 2 — capital over time for one coin, numbered as in chart 1
# ============================================================================
COIN_RANK = 80                  # position in chart 1: 1 = least profitable, len(summary) = most

if not 1 <= COIN_RANK <= len(summary):
    raise ValueError(f'COIN_RANK must be between 1 and {len(summary)}')
row = summary.iloc[COIN_RANK - 1]
r = results[row['coin']]
t = pd.to_datetime(r['timestamps'], utc=True).tz_convert(None)
start = INITIAL_CAPITAL_USD
# At the last minute an open position is sold (as strategy.ipynb), so the final values are
# all cash. The chart is (1 + TAIL) times as wide as without the tail (TAIL in the config), and the extra part
# holds the final state after the end of the simulation, whatever the coin's length.
sim_end = t[-1]
tail = (sim_end - t[0]) * TAIL
t = t.append(pd.DatetimeIndex([sim_end + tail]))
hold = lambda a: np.append(a, a[-1])
cash, openv, cum_fees = hold(r['cash']), hold(r['open']), hold(r['cum_fees'])
closed = np.minimum(cash, start)
realized = np.maximum(cash - start, 0.0)

fig, (axf, axc) = plt.subplots(2, 1, figsize=CHART_SIZE, sharex=True, gridspec_kw={'height_ratios': [1, 4]})
fig.subplots_adjust(hspace=0.06, **CHART_MARGINS)
axf.fill_between(t, 0, cum_fees, step='post', color=THEME['fees'], alpha=0.35, linewidth=0)
axf.step(t, cum_fees, where='post', color=THEME['fees'], linewidth=1)
axf.set_ylabel('Fees')
axf.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:.2f}'))
axf.set_ylim(0, max(cum_fees.max() * 1.15, 0.01))

b1, b2, b3 = closed, closed + realized, closed + realized + openv
axc.fill_between(t, 0, b1, step='post', color=THEME['capital'], alpha=0.45, linewidth=0, label='Closed capital')
axc.fill_between(t, b1, b2, step='post', color=THEME['realized'], alpha=0.5, linewidth=0, label='Realized profit')
axc.fill_between(t, b2, b3, step='post', color=THEME['open'], alpha=0.45, linewidth=0, label='Open position')
for ax in (axf, axc):
    ax.axvline(sim_end, color=THEME['text'], linestyle='--', linewidth=0.7,
               label='End of simulation (open position sold)' if ax is axc else None)
axc.set_ylim(0, max(b3.max(), start) * 1.18)
# the final split, written in the extra day after the end
final_cash = cash[-1]
split = (f'Final ${final_cash:.2f}\n${closed[-1]:.2f} capital\n+ ${realized[-1]:.2f} profit'
         if realized[-1] > 0 else f'Final ${final_cash:.2f}\n(loss: ${start - final_cash:.2f})')
axc.annotate(tx(split), xy=(sim_end + tail / 2, b2[-1]), xytext=(0, 8),
             textcoords='offset points', ha='center', va='bottom', fontsize=9.5, color=THEME['text'])
axc.set_ylabel('Capital')
axc.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:.0f}'))
_loc = mdates.AutoDateLocator()
axc.xaxis.set_major_locator(_loc)
axc.xaxis.set_major_formatter(mdates.ConciseDateFormatter(_loc))
axc.set_xlabel('Date (UTC)')
axc.set_xlim(t[0], t[-1])
axc.legend(loc='upper left', ncol=3, frameon=True, fontsize=10)
coin_label = row['coin'].removesuffix('_cleaned')
axf.set_title(tx(f"Coin {COIN_RANK} of {len(summary)}, least to most profitable: {coin_label}   final ${row['final_capital']:.2f}, "
                 f"{int(row['trades'])} trades, {int(row['rebases'])} rebases, ${row['fees']:.2f} fees, {row['days']:.1f} days"),
              loc='left', fontsize=12, pad=10)
fig.align_ylabels((axf, axc))
path = OUT_DIR / f'capital_over_time_rank_{COIN_RANK}.png'
fig.savefig(path, dpi=200)
plt.show()
print(f'saved {path}')

# the neighbouring ranks, to help pick a coin
lo, hi = max(0, COIN_RANK - 6), min(len(summary), COIN_RANK + 5)
summary.iloc[lo:hi][['rank', 'coin', 'final_capital', 'trades', 'rebases', 'fees', 'days']]

## Feature importance of the chosen models

Retrains the move and the direction model in `live_pipeline/artifacts/` from their saved parameters (`move.json`, `direction.json`: the models' `_params.json` from `model_search`), exactly as `model_search` trained them, and shows which feature families they rely on. Nothing is saved.

* **Move model:** the `train` split, rows with at least 240 minutes of future, early stopping on `val_train`.
* **Direction model:** the same splits, but only the move model's candidate minutes (tradable 12 h after listing, both price positions ≤ 0.575, move score in its top 50% on train, outcome decided), with the move score of `artifacts/move.ubj`, which defined these rows in `model_search`.
* **Check:** the retrained models' best iteration and validation AUC are compared with the saved ones.
* **Importance:** each feature's share of the model's total gain (the summed loss reduction of all its splits), added up per feature family.

Needs the config and the data cells above (the models and the strategy file); the strategy simulation is not needed.

In [ ]:

# ============================================================================
# RETRAIN THE CHOSEN MODELS (not saved) — as model_search trained them
# ============================================================================
from sklearn.metrics import roc_auc_score

DEVICE = None                   # None: as in model_search (its fixed_params: 'cuda'); 'cpu' without an NVIDIA GPU

# model_search's training settings (not in the saved parameters)
SPLITS_DIR = REPO_DATA / 'dataset_v9' / 'splits'
SPLIT_CACHE = REPO_RESULTS / 'model_search'     # model_search's parquet copies of the splits, used if present
MOVE_LABEL, DIR_LABEL, META_COL = 'label_10pct_30m', 'label_fpclass_15pct_2h', 'label_meta_minutes_to_end'
MIN_MINUTES_TO_END = 240
N_ESTIMATORS_MAX, EARLY_STOPPING_ROUNDS = 5000, 100
CAND_FRAC, TRADING_START_MIN, A50_PP_CAP = 0.50, 12 * 60, 0.575


def _saved_params(p):
    """The model's _params.json from model_search, copied next to it as <name>.json."""
    for pj in (p.with_name(p.stem + '_params.json'), p.with_suffix('.json')):
        if pj.exists():
            with open(pj) as f:
                return json.load(f)
    raise RuntimeError(f"no parameters for {p.name}: expected {p.with_suffix('.json').name} next to it")


SAVED = {k: _saved_params(_found[k][0]) for k in ('move', 'dir')}
FEATS = {k: list(MODELS[k].feature_names) for k in ('move', 'dir')}   # each model's own features, in order


def _load_split(name):
    cache = SPLIT_CACHE / f'cache_{name}.parquet'
    if cache.exists():
        df = pd.read_parquet(cache)
    else:
        df = pd.read_csv(SPLITS_DIR / f'{name}.csv')
        df['coin_id'] = df['coin_id'].astype(str)
        df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', utc=True)
    return df[df[META_COL] >= MIN_MINUTES_TO_END].reset_index(drop=True)


def _ohlcv_positions(df):
    """Position of every row in its coin's OHLCV timeline (-1 if not found)."""
    pos = np.full(len(df), -1, dtype=np.int64)
    for coin, idx in df.groupby('coin_id', sort=False).indices.items():
        f = list(OHLCV_FOLDER.glob(f'{coin}_ohlcv_cleaned.csv')) or \
            list(OHLCV_FOLDER.glob(f"{coin.replace('_cleaned', '')}_ohlcv_cleaned.csv"))
        if not f:
            continue
        ts = pd.to_datetime(pd.read_csv(f[0], usecols=['timestamp'])['timestamp'], format='mixed', utc=True, errors='coerce')
        ts = np.sort(ts[ts.notna()].to_numpy())
        q = df['timestamp'].to_numpy()[idx]
        p = np.searchsorted(ts, q)
        ok = (p < len(ts)) & (ts[np.minimum(p, len(ts) - 1)] == q)
        pos[idx[ok]] = p[ok]
    return pos


def _candidates(df, move_score, threshold):
    """A50 direction rows: tradable, move score >= threshold, outcome decided."""
    tradable = ((_ohlcv_positions(df) >= TRADING_START_MIN) & (df['price_position_15m'].to_numpy() <= A50_PP_CAP)
                & (df['price_position_1h'].to_numpy() <= A50_PP_CAP))
    return tradable & (move_score >= threshold) & df[DIR_LABEL].isin([1, 2]).to_numpy()


def _train(kind, tr, y_tr, va, y_va):
    saved = SAVED[kind]
    params = dict(saved['fixed_params'], **saved['tuned_params'])
    params['scale_pos_weight'] = saved['scale_pos_weight']
    if DEVICE:
        params['device'] = DEVICE
    f = FEATS[kind]
    dtr = xgb.DMatrix(tr[f], label=y_tr, feature_names=f)
    dva = xgb.DMatrix(va[f], label=y_va, feature_names=f)
    es = xgb.callback.EarlyStopping(rounds=EARLY_STOPPING_ROUNDS, min_delta=1e-4, save_best=True,
                                    maximize=True, metric_name='auc')
    model = xgb.train(params, dtr, num_boost_round=N_ESTIMATORS_MAX, evals=[(dva, 'validation_0')],
                      callbacks=[es], verbose_eval=False)
    auc = roc_auc_score(y_va, model.predict(dva))
    print(f"  {kind:<4} {len(tr):>10,} rows: best iteration {model.best_iteration} (saved model "
          f"{saved.get('best_iteration')}), val AUC {auc:.4f} (saved {saved.get('auc', float('nan')):.4f})")
    return model


train_df, val_df = _load_split('train'), _load_split('val_train')
print(f"train {len(train_df):,} rows, val_train {len(val_df):,} rows (minutes_to_end >= {MIN_MINUTES_TO_END})")
TRAINED = {}
TRAINED['move'] = _train('move', train_df, train_df[MOVE_LABEL].astype(int),
                         val_df, val_df[MOVE_LABEL].astype(int).to_numpy())

# direction rows: the candidates of the move model in the artifacts folder (as in model_search)
pm_tr, pm_va = predict(MODELS['move'], train_df), predict(MODELS['move'], val_df)
_thr = float(np.quantile(pm_tr, 1 - CAND_FRAC))
a_tr = train_df[_candidates(train_df, pm_tr, _thr)].reset_index(drop=True)
a_va = val_df[_candidates(val_df, pm_va, _thr)].reset_index(drop=True)
TRAINED['dir'] = _train('dir', a_tr, (a_tr[DIR_LABEL] == 2).astype(int),
                        a_va, (a_va[DIR_LABEL] == 2).astype(int).to_numpy())
del train_df, val_df, a_tr, a_va


In [ ]:

# ============================================================================
# FEATURE IMPORTANCE PER FAMILY — one image with a pie chart per model (shown, not saved)
# ============================================================================
# Feature families of Dataset_creationV9, as in the README's feature list. The pump set of a model is
# PUMP_FEATURES (move) or the same names + '_p15_2h' (direction).
PUMP_FEATURES = ['cooling_off_indicator', 'excess_buying_pressure_4h', 'cumulative_positive_since_last_pump',
    'buying_pressure_accumulation_rate', 'recent_pump_rate_7d', 'pump_density_1h', 'pump_density_2h', 'pump_density_4h',
    'pump_acceleration_4h', 'pump_clustering_coef', 'time_since_cluster', 'pump_velocity_8h', 'time_weighted_pump_density',
    'minutes_since_last_pump', 'last_pump_magnitude', 'pump_failure_rate_7d', 'time_between_pumps_mean',
    'time_between_pumps_std', 'liquidity_change_since_pump', 'pump_velocity_per_liquidity', 'liquidity_weighted_pump_density',
    'pump_time_liquidity_staleness', 'cooling_off_liquidity_adjusted']
FAMILY_RULES = [
    ('Pump dynamics', lambda f: f in PUMP_FEATURES or f.endswith('_p15_2h')),
    ('Lifecycle & time', lambda f: f in ('minutes_since_launch', 'hour_sin', 'hour_cos') or f.startswith(('launch_', 'price_stability_'))),
    ('Liquidity', lambda f: 'liquidity' in f or f == 'estimated_sol_1pct_impact'),
    ('Volume', lambda f: f.startswith('volume_') or f in ('delta_v_acceleration',)),
    ('Volatility', lambda f: f.startswith(('volatility', 'vol_stability', 'intraday_vol', 'bb_width', 'hl_', 'range_compression',
                                           'downside_deviation', 'upside_deviation', 'avg_candle_body', 'gap_frequency',
                                           'loss_tail', 'gain_tail', 'return_skewness', 'return_kurtosis', 'max_single_',
                                           'gain_loss_ratio'))),
    ('Momentum', lambda f: f.startswith(('return_', 'momentum', 'risk_adj_momentum', 'vol_norm_momentum', 'drawdown', 'drawup',
                                         'minutes_since_ath', 'price_position', 'distance_to', 'price_to_ma', 'ma_crossover',
                                         'consecutive_', 'sum_positive', 'sum_negative', 'price_efficiency', 'upside_capture'))),
]
def family_of(f):
    for name, rule in FAMILY_RULES:
        if rule(f):
            return name
    return 'Other'


MODEL_TITLES = {'move': 'Move model', 'dir': 'Direction model'}
FAMILY_ORDER = ['Momentum', 'Volatility', 'Pump dynamics', 'Liquidity', 'Volume', 'Lifecycle & time', 'Other']
# clearly different hues, so neighbouring slices can't be confused
FAMILY_COLORS = {'Momentum': '#3b82f6', 'Volatility': '#fbbf24', 'Pump dynamics': '#22c55e', 'Liquidity': '#22d3ee',
                 'Volume': '#ef4444', 'Lifecycle & time': '#a855f7', 'Other': '#94a3b8'}
PIE_FIG_SIZE = (CHART_SIZE[0] * 2 / 3, CHART_SIZE[1] * 2 / 3)    # 2/3 of the other charts' size

rows = []
for kind, model in TRAINED.items():
    gain = model.get_score(importance_type='total_gain')          # features never used in a split have none
    total = sum(gain.values()) or 1.0
    for f in FEATS[kind]:
        rows.append({'model': kind, 'feature': f, 'family': family_of(f), 'share': gain.get(f, 0.0) / total})
IMP = pd.DataFrame(rows)
if (IMP['family'] == 'Other').any():
    print('features without a family:', sorted(IMP.loc[IMP['family'] == 'Other', 'feature'].unique()))

fam = IMP.groupby(['model', 'family']).agg(features=('feature', 'size'), share=('share', 'sum')).reset_index()
families = [f for f in FAMILY_ORDER if f in set(fam['family'])]

with plt.rc_context({'axes.grid': False}):   # the dark theme of the config cell, without a grid
    # pie | legend | pie: the legend sits between the charts, so the pies can use the full height
    fig = plt.figure(figsize=PIE_FIG_SIZE)
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 0.62, 1], wspace=0.0)
    axes = {'move': fig.add_subplot(gs[0]), 'dir': fig.add_subplot(gs[2])}
    ax_leg = fig.add_subplot(gs[1])
    ax_leg.axis('off')
    for kind, ax in axes.items():
        d = fam[fam['model'] == kind].set_index('family').reindex(families).fillna(0)
        wedges, _, pct = ax.pie(
            d['share'], colors=[FAMILY_COLORS[f] for f in families], startangle=90, counterclock=False,
            radius=1.15, autopct=lambda p: f'{p:.0f}%' if p >= 3 else '', pctdistance=0.7,
            wedgeprops={'linewidth': 1.2, 'edgecolor': THEME['figure']},
            textprops={'color': THEME['figure'], 'fontsize': 9, 'fontweight': 'bold'})
        ax.set_title(f"{MODEL_TITLES[kind]} ({int(d['features'].sum())} features)", fontsize=10.5, pad=6)
        ax.set_aspect('equal')
    # one legend for both: each family with its number of features (the same in both models)
    n_feat = fam[fam['model'] == 'move'].set_index('family')['features']
    labels = [f"{f} ({int(n_feat.get(f, 0))})" for f in families]
    ax_leg.legend(wedges, labels, loc='center', frameon=False, fontsize=9, title='Family (features)',
                  title_fontsize=9, labelcolor=THEME['text'], handlelength=1.2, borderaxespad=0)
    ax_leg.get_legend().get_title().set_color(THEME['muted'])
    fig.suptitle('Feature importance by family (share of the total gain)', x=0.015, ha='left', fontsize=11.5, y=0.98)
    fig.text(0.015, 0.895, "Pump dynamics: each model's own pump set (move: +10% in 30 min; direction: +15% in 2 h).",
             fontsize=8, color=THEME['muted'])
    fig.subplots_adjust(left=0.01, right=0.99, top=0.80, bottom=0.03)
    plt.show()

# the numbers behind the charts, and each model's ten most important features
table = fam.pivot(index='family', columns='model', values='share').reindex(families)
table.insert(0, 'features', n_feat.reindex(families).astype(int))
table = table.rename(columns={'move': 'move share', 'dir': 'direction share'})
display(table.style.format({'move share': '{:.1%}', 'direction share': '{:.1%}'}))
for kind in ('move', 'dir'):
    top = IMP[IMP['model'] == kind].nlargest(10, 'share')[['feature', 'family', 'share']]
    print(f"\n{MODEL_TITLES[kind]}: ten most important features")
    print(top.to_string(index=False, formatters={'share': '{:.1%}'.format}))
